In [2]:
import os
import base64
from dotenv import load_dotenv
import anthropic

# 1. .env dosyasından API anahtarını yükle
load_dotenv(dotenv_path='../.env')
api_key = os.getenv("ANTHROPIC_API_KEY")

if not api_key:
    print("Hata: API anahtarı bulunamadı. .env dosyanı kontrol et.")
else:
    print("API anahtarı başarıyla yüklendi.")

# 2. Anthropic istemcisini (client) başlat
client = anthropic.Anthropic(api_key=api_key)

# 3. Görseli Base64 formatına çeviren fonksiyon (API'ler görseli bu formatta ister)
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# 4. Hazırladığımız test belgesinin yolu
image_path = "../data/raw_docs/test_talep_01.png"

# 5. Görseli dönüştür
try:
    base64_image = encode_image(image_path)
    print("Görsel başarıyla Base64 formatına çevrildi.")
except FileNotFoundError:
     print(f"Hata: Görsel bulunamadı. Lütfen {image_path} yolunu kontrol et.")

# 6. Görseli Claude'a gönder (Multimodal bağlantı)
try:
    print("Claude'a istek gönderiliyor, lütfen bekle...")
    response = client.messages.create(
        model="claude-sonnet-5", # settings.yaml'da belirlediğimiz model
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": "image/png",
                            "data": base64_image,
                        },
                    },
                    {
                        "type": "text",
                        "text": "Lütfen bu görseldeki metni birebir okur musun?"
                    }
                ],
            }
        ],
    )
    print("\n--- Claude'un Yanıtı ---")
    # Yanıt birden fazla blok içerebilir (ör. thinking bloğu); sadece metin bloklarını yazdır.
    for block in response.content:
        if block.type == "text":
            print(block.text)
except Exception as e:
    print(f"API isteği sırasında bir hata oluştu: {e}")


API anahtarı başarıyla yüklendi.
Görsel başarıyla Base64 formatına çevrildi.
Claude'a istek gönderiliyor, lütfen bekle...

--- Claude'un Yanıtı ---
İşte görseldeki metnin birebir okunuşu:

```
DONANIM TALEP FORMU

Talep Eden: Dila Alpay
Tarih: 12.08.2026
Departman: Yazılım Geliştirme
Konu: Ek Monitör Talebi

Açıklama:
Geliştirmekte olduğum yapay zeka destekli doküman analiz modülünün test işlemleri sırasında, log takibi ve kod yazımını eşzamanlı yürütebilmek amacıyla tarafıma 1 adet 27 inç monitör tahsis edilmesini rica ederim.

İmza:
```
